# Query RHOAI 3.4 product documentation with Stage 230 RAG

This notebook uses the repo-stored official Red Hat OpenShift AI 3.4 PDF guide corpus, prepares metadata-rich chunks, indexes them through the same Llama Stack Files and Vector Stores path used by the Stage 230 RAG runtime, and validates grounded answers with Nemotron through Models-as-a-Service.

## 1. Review the selected official sources

The source manifest covers Llama Stack RAG, AutoRAG, RAGAS, EvalHub, guardrails, AI Pipelines, and Docling data preparation. The stage commits the selected official PDFs under `.stage230/data/rhoai-product-docs/source` for deterministic demo rebuilds. Use the preparation helper's `--force-download` option only when intentionally refreshing the corpus from `docs.redhat.com`.

In [ ]:
import json
from pathlib import Path

manifest_path = Path('.stage230/data/rhoai-product-docs/metadata/rhoai-3.4-product-docs.json')
manifest = json.loads(manifest_path.read_text())
for document in manifest['documents']:
    print(f"{document['guide_slug']}: {document['title']} -> {document['source_url']}")

## 2. Download and prepare focused chunks

The preparation helper reads the staged PDFs, extracts text with `pypdf`, keeps focused chunks around the topics in the manifest, and writes JSONL with product, version, guide, page, topic, and source URL metadata.

In [ ]:
import subprocess
import sys

subprocess.run([
    sys.executable,
    '.stage230/scripts/rhoai_product_docs_prepare.py',
    '--manifest', '.stage230/data/rhoai-product-docs/metadata/rhoai-3.4-product-docs.json',
    '--source-dir', '.stage230/data/rhoai-product-docs/source',
    '--output', '.stage230/data/rhoai-product-docs/processed/rhoai-3.4-product-docs-chunks.jsonl',
], check=True)

## 3. Ingest and ask demo-audience questions

This smoke path creates or resets the `stage230-rhoai-34-product-docs` vector store, uploads prepared chunks, verifies metadata-filtered hybrid retrieval, reranks candidates through Llama Stack, and asks Nemotron to answer from the retrieved official documentation context.

In [ ]:
subprocess.run([
    sys.executable,
    '.stage230/scripts/rhoai_product_docs_rag_smoke.py',
    '--reset',
    '--manifest', '.stage230/data/rhoai-product-docs/metadata/rhoai-3.4-product-docs.json',
    '--sample', '.stage230/data/rhoai-product-docs/processed/rhoai-3.4-product-docs-chunks.jsonl',
    '--vector-store', 'stage230-rhoai-34-product-docs',
    '--max-questions', '3',
    '--search-mode', 'hybrid',
], check=True)

## 4. Ask a focused question

Use this pattern during the demo to explain one capability at a time. Change `--query` and `--expected-topic` to `llama_stack_rag`, `autorag`, `ragas_evaluation`, `evalhub`, `guardrails`, `ai_pipelines`, or `docling_data_prep`.

In [ ]:
subprocess.run([
    sys.executable,
    '.stage230/scripts/rhoai_product_docs_rag_smoke.py',
    '--manifest', '.stage230/data/rhoai-product-docs/metadata/rhoai-3.4-product-docs.json',
    '--sample', '.stage230/data/rhoai-product-docs/processed/rhoai-3.4-product-docs-chunks.jsonl',
    '--vector-store', 'stage230-rhoai-34-product-docs',
    '--query', 'How does OpenShift AI use Llama Stack, vector stores, and hybrid search for RAG?',
    '--expected-topic', 'llama_stack_rag',
    '--expected-term', 'Llama Stack',
    '--expected-term', 'vector',
    '--search-mode', 'hybrid',
], check=True)